In [9]:
import pandas as pd
import numpy as np
from collections import defaultdict

In [10]:
root_dir = "/data1/shared_datasets/project-training-data/olmo3/"
files = [("Dolma3", f"{root_dir}dolma3/dolma3_left_right.csv"), ("Dolmino3", f"{root_dir}dolmino3/dolmino_classified_left_right.csv"),("SFT3", f"{root_dir}sft/sft_left_right.csv"), ("DPO3", f"{root_dir}dpo/dpo_left_right.csv"), ("RefinedWeb",  f"{root_dir}refinedweb/refinedweb_classified_left_right.csv"), 
        ("Pile", f"{root_dir}pile/pile_classified_left_right.csv")]

In [11]:
proportions = {}
counts = {}
ratios = []
for model_name, file_path in files:
    df = pd.read_csv(file_path)
    if "pile" in file_path or "refinedweb" in file_path:
        label_left = "left"
        label_right = "right"
    else:
        label_left = "1"
        label_right = "3"
    column_name = "prediction"
    df[column_name] = df[column_name].astype(str)
    left_count = (df[column_name] == label_left).sum()
    right_count = (df[column_name] == label_right).sum()
    print(left_count, right_count)
    neutral = pd.read_csv(file_path.replace("left_right", "neutral"))
    neutral_count = len(neutral)
    total_count = left_count + right_count + neutral_count

    counts[model_name] = (int(left_count), int(right_count), int(neutral_count))

    left_proportion = left_count / total_count if total_count > 0 else 0
    right_proportion = right_count / total_count if total_count > 0 else 0
    proportions[model_name] = left_proportion + right_proportion

    print(f"Model: {model_name}")
    print(f"Ratio: {left_count / right_count if right_count > 0 else float('inf')}")
    ratios.append(left_count / right_count if right_count > 0 else float('inf'))
    print(f"Left Proportion: {left_proportion:.2%}")
    print(f"Right Proportion: {right_proportion:.2%}")
    print(f"Total Count: {total_count}")
    print("-" * 30)

18196 10260
Model: Dolma3
Ratio: 1.7734892787524366
Left Proportion: 3.84%
Right Proportion: 2.17%
Total Count: 473837
------------------------------
5758 3472
Model: Dolmino3
Ratio: 1.6584101382488479
Left Proportion: 3.06%
Right Proportion: 1.85%
Total Count: 188107
------------------------------
3289 523
Model: SFT3
Ratio: 6.288718929254302
Left Proportion: 2.27%
Right Proportion: 0.36%
Total Count: 144927
------------------------------
4205 941
Model: DPO3
Ratio: 4.468650371944739
Left Proportion: 1.88%
Right Proportion: 0.42%
Total Count: 223736
------------------------------
6089 2676
Model: RefinedWeb
Ratio: 2.2754110612855007
Left Proportion: 3.13%
Right Proportion: 1.38%
Total Count: 194335
------------------------------
7807 3060
Model: Pile
Ratio: 2.5513071895424835
Left Proportion: 4.46%
Right Proportion: 1.75%
Total Count: 174849
------------------------------


In [12]:
np.mean(ratios), np.std(ratios)

(3.1693311615047186, 1.6748688571478327)

In [13]:
for corpora in [("DPO3", "Dolmino3"), ("DPO3", "RefinedWeb"), ("DPO3", "Pile"),
                ("SFT3", "Dolmino3"), ("SFT3", "RefinedWeb"), ("SFT3", "Pile"), 
                ("SFT3", "Dolma3"), ("DPO3", "Dolma3")]:
    print(corpora[0], "vs", corpora[1])
    print(proportions[corpora[1]] / proportions[corpora[0]])

DPO3 vs Dolmino3
2.1333535328111237
DPO3 vs RefinedWeb
1.9609520907853302
DPO3 vs Pile
2.7021695875513054
SFT3 vs Dolmino3
1.8654909832273743
SFT3 vs RefinedWeb
1.7147361595902788
SFT3 vs Pile
2.3628868460849177
SFT3 vs Dolma3
2.283185998977015
DPO3 vs Dolma3
2.6110246368255052


In [14]:
np.mean([proportions["SFT3"], proportions["DPO3"]]) * 100, np.std([proportions["SFT3"], proportions["DPO3"]])* 100

(2.4651609564282526, 0.16512877564375633)

In [15]:
np.mean([proportions["Dolma3"], proportions["Dolmino3"], proportions["Pile"], proportions["RefinedWeb"]]) * 100, np.std([proportions["Dolma3"], proportions["Dolmino3"], proportions["Pile"], proportions["RefinedWeb"]]) * 100

(5.409388097765942, 0.7185872434159275)

In [18]:
# Bootstrap significance tests.
# Each doc is left / right / neutral, so resampling docs with replacement == Multinomial(N, [pL, pR, pN]).
# Corpora are independent -> bootstrapped independently. p-values are two-sided (null: diff=0 / ratio=1).
B = 10_000
rng = np.random.default_rng(0)

def ci(x, alpha=0.05):
    return np.percentile(x, [100 * alpha / 2, 100 * (1 - alpha / 2)])

def pval(x, null):
    return min(1.0, 2 * min((np.sum(x <= null) + 1) / (len(x) + 1), (np.sum(x >= null) + 1) / (len(x) + 1)))

def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for i, idx in enumerate(order):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(1, run)
    return adj

boots = {}
for name, (l, r, n) in counts.items():
    N = l + r + n
    d = rng.multinomial(N, [l / N, r / N, n / N], size=B)
    boots[name] = (d[:, 0] / N, d[:, 1] / N)  # bootstrap left prop, right prop

# 1) Within each corpus: is left different from right?
rows = []
for name, (l, r, n) in counts.items():
    L, R = boots[name]; N = l + r + n
    rows.append({
        "corpus": name, "N": N,
        "left %": 100 * l / N, "left 95% CI": tuple(np.round(100 * ci(L), 2)),
        "right %": 100 * r / N, "right 95% CI": tuple(np.round(100 * ci(R), 2)),
        "political %": 100 * (l + r) / N, "political 95% CI": tuple(np.round(100 * ci(L + R), 2)),
        "L/R ratio": l / r, "L/R 95% CI": tuple(np.round(ci(L / R), 2)),
        "p (left=right)": pval(L - R, 0),
    })
within = pd.DataFrame(rows).round(3)
display(within)

# # 2) Between corpora (same pairs as above): ratio of political share, and ratio of L/R ratios
# pairs = [("DPO3", "Dolmino3"), ("DPO3", "RefinedWeb"), ("DPO3", "Pile"), ("DPO3", "Dolma3"),
#          ("SFT3", "Dolmino3"), ("SFT3", "RefinedWeb"), ("SFT3", "Pile"), ("SFT3", "Dolma3")]
# rows = []
# for post, pre in pairs:
#     Lp, Rp = boots[post]; Lq, Rq = boots[pre]
#     pol_ratio, lr_ratio = (Lq + Rq) / (Lp + Rp), (Lq / Rq) / (Lp / Rp)
#     rows.append({
#         "post-training": post, "pretraining": pre,
#         "political ratio": proportions[pre] / proportions[post],
#         "political ratio 95% CI": tuple(np.round(ci(pol_ratio), 2)),
#         "p (political)": pval((Lq + Rq) - (Lp + Rp), 0),
#         "L/R skew ratio": (counts[pre][0] / counts[pre][1]) / (counts[post][0] / counts[post][1]),
#         "L/R skew 95% CI": tuple(np.round(ci(lr_ratio), 2)),
#         "p (L/R skew)": pval(lr_ratio, 1),
#     })
# between = pd.DataFrame(rows)
# between["p (political) Holm"] = holm(between["p (political)"])
# between["p (L/R skew) Holm"] = holm(between["p (L/R skew)"])
# display(between.round(4))

# # 3) Pooled: mean political share of pretraining corpora vs post-training (SFT3, DPO3)
# pol = lambda names: np.mean([boots[n][0] + boots[n][1] for n in names], axis=0)
# pre_b, post_b = pol(["Dolma3", "Dolmino3", "Pile", "RefinedWeb"]), pol(["SFT3", "DPO3"])
# print(f"pretraining political share:   {100*pre_b.mean():.2f}% CI {np.round(100*ci(pre_b), 2)}")
# print(f"post-training political share: {100*post_b.mean():.2f}% CI {np.round(100*ci(post_b), 2)}")
# print(f"ratio pre/post: {(pre_b/post_b).mean():.2f} CI {np.round(ci(pre_b/post_b), 2)}, p={pval(pre_b - post_b, 0):.4g}")


,corpus,N,left %,left 95% CI,right %,right 95% CI,political %,political 95% CI,L/R ratio,L/R 95% CI,p (left=right)
0,Dolma3,473837,3.840,"(3.79, 3.9)",2.165,"(2.12, 2.21)",6.005,"(5.94, 6.07)",1.773,"(1.73, 1.82)",0.0
1,Dolmino3,188107,3.061,"(2.98, 3.14)",1.846,"(1.79, 1.91)",4.907,"(4.81, 5.0)",1.658,"(1.59, 1.73)",0.0
2,SFT3,144927,2.269,"(2.19, 2.35)",0.361,"(0.33, 0.39)",2.630,"(2.55, 2.72)",6.289,"(5.75, 6.9)",0.0
3,DPO3,223736,1.879,"(1.82, 1.93)",0.421,"(0.39, 0.45)",2.300,"(2.24, 2.36)",4.469,"(4.17, 4.8)",0.0
4,RefinedWeb,194335,3.133,"(3.06, 3.21)",1.377,"(1.33, 1.43)",4.510,"(4.42, 4.6)",2.275,"(2.17, 2.38)",0.0
5,Pile,174849,4.465,"(4.37, 4.56)",1.750,"(1.69, 1.81)",6.215,"(6.1, 6.33)",2.551,"(2.45, 2.66)",0.0


In [21]:
# push dataset to hub
from datasets import load_dataset

local_dir = "/home/ceron/political_content_training_data_private/stance_detection/annotations/dolma_tacl_sample_gold_cleaned_reannotated.csv"
repo_id = "tceron/stancepol"
dataset = load_dataset("csv", data_files=local_dir)
dataset.push_to_hub(repo_id)


Generating train split: 0 examples [00:00, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/tceron/stancepol/commit/5eb7c3b25fb9acdd08bb84c142e3355e9d677cb9', commit_message='Upload dataset', commit_description='', oid='5eb7c3b25fb9acdd08bb84c142e3355e9d677cb9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/tceron/stancepol', endpoint='https://huggingface.co', repo_type='dataset', repo_id='tceron/stancepol'), pr_revision=None, pr_num=None)